<a href="https://colab.research.google.com/github/Josh012006/OpenX-Embodiment-Datasets-Visualization/blob/main/colabs/Open_X_Embodiment_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright 2020 DeepMind Technologies Limited.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Open X-Embodiment Datasets

![](https://robotics-transformer-x.github.io/img/overview.png)

This colab helps you **visualize** the datasets in the Open X-Embodiment Dataset end effectors positions for the training episodes.

# Visualize Datasets

In [ ]:
# Install dependencies
%pip install tensorflow tensorflow-datasets matplotlib numpy gcsfs ipywidgets plotly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import tensorflow_datasets as tfds
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

DATASETS = [
    'fractal20220817_data',
    'kuka',
    'bridge',
    'taco_play',
    'jaco_play',
    'berkeley_cable_routing',
    'roboturk',
    'nyu_door_opening_surprising_effectiveness',
    'viola',
    'berkeley_autolab_ur5',
    'toto',
    'language_table',
    'columbia_cairlab_pusht_real',
    'stanford_kuka_multimodal_dataset_converted_externally_to_rlds',
    'nyu_rot_dataset_converted_externally_to_rlds',
    'stanford_hydra_dataset_converted_externally_to_rlds',
    'austin_buds_dataset_converted_externally_to_rlds',
    'nyu_franka_play_dataset_converted_externally_to_rlds',
    'maniskill_dataset_converted_externally_to_rlds',
    'cmu_franka_exploration_dataset_converted_externally_to_rlds',
    'ucsd_kitchen_dataset_converted_externally_to_rlds',
    'ucsd_pick_and_place_dataset_converted_externally_to_rlds',
    'austin_sailor_dataset_converted_externally_to_rlds',
    'austin_sirius_dataset_converted_externally_to_rlds',
    'bc_z',
    'usc_cloth_sim_converted_externally_to_rlds',
    'utokyo_pr2_opening_fridge_converted_externally_to_rlds',
    'utokyo_pr2_tabletop_manipulation_converted_externally_to_rlds',
    'utokyo_saytap_converted_externally_to_rlds',
    'utokyo_xarm_pick_and_place_converted_externally_to_rlds',
    'utokyo_xarm_bimanual_converted_externally_to_rlds',
    'robo_net',
    'berkeley_mvp_converted_externally_to_rlds',
    'berkeley_rpt_converted_externally_to_rlds',
    'kaist_nonprehensile_converted_externally_to_rlds',
    'stanford_mask_vit_converted_externally_to_rlds',
    'tokyo_u_lsmo_converted_externally_to_rlds',
    'dlr_sara_pour_converted_externally_to_rlds',
    'dlr_sara_grid_clamp_converted_externally_to_rlds',
    'dlr_edan_shared_control_converted_externally_to_rlds',
    'asu_table_top_converted_externally_to_rlds',
    'stanford_robocook_converted_externally_to_rlds',
    'eth_agent_affordances',
    'imperialcollege_sawyer_wrist_cam',
    'iamlab_cmu_pickup_insert_converted_externally_to_rlds',
    'uiuc_d3field',
    'utaustin_mutex',
    'berkeley_fanuc_manipulation',
    'cmu_play_fusion',
    'cmu_stretch',
    'berkeley_gnm_recon',
    'berkeley_gnm_cory_hall',
    'berkeley_gnm_sac_son'
]

In [ ]:
# Utils
def dataset2path(dataset_name):
  if dataset_name == 'robo_net':
    version = '1.0.0'
  elif dataset_name == 'language_table':
    version = '0.0.1'
  else:
    version = '0.1.0'
  return f'gs://gresearch/robotics/{dataset_name}/{version}'

def extract_endpoint(step, config):
    """Extrait la position x,y,z du end effector depuis un step."""
    data = step
    for key in config["field"]:
        data = data[key]
    data = data.numpy()

    if config["reshape"]:
        flat = data.flatten()
        if len(flat) >= 16:
            matrix = flat[:16].reshape(4, 4)
            return matrix[:3, 3]
        else:
            raise ValueError(f"Cannot reshape data of size {len(flat)} into 4x4 matrix")
    elif config["indices"] is not None:
        return data[config["indices"]]
    else:
        return data

def get_safe_split(b, max_episodes=500):
    """Retourne un split sûr selon la taille réelle du dataset."""
    try:
        total = b.info.splits['train'].num_examples
        n = min(max_episodes, total)
        return f'train[:{n}]'
    except Exception:
        return f'train[:{max_episodes}]'

def normalize(endpoints):
    """Normalise les endpoints entre 0 et 1 par axe."""
    mins = endpoints.min(axis=0)
    maxs = endpoints.max(axis=0)
    ranges = maxs - mins
    ranges[ranges == 0] = 1
    return (endpoints - mins) / ranges

## End effector coordinates fields

Visualizing the available features and record the end effector fields for each dataset where it is available.

In [ ]:
for dataset_name in DATASETS:
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_name}")
    print('='*60)
    try:
        b = tfds.builder_from_directory(builder_dir=dataset2path(dataset_name))
        print(b.info.features)
    except Exception as e:
        print(f"ERROR: {e}")

In [ ]:
DATASET_EEF_CONFIG = {
    "fractal20220817_data": {
        "field": ["observation", "base_pose_tool_reached"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "kuka": {
        "field": ["observation", "clip_function_input/base_pose_tool_reached"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "jaco_play": {
        "field": ["observation", "end_effector_cartesian_pos"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "taco_play": {
        "field": ["observation", "robot_obs"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "maniskill_dataset_converted_externally_to_rlds": {
        "field": ["observation", "tcp_pose"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "stanford_kuka_multimodal_dataset_converted_externally_to_rlds": {
        "field": ["observation", "ee_position"],
        "indices": None,
        "reshape": False
    },
    "nyu_rot_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "stanford_hydra_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "nyu_franka_play_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(7, 10),
        "reshape": False
    },
    "austin_sailor_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "bc_z": {
        "field": ["observation", "present/xyz"],
        "indices": None,
        "reshape": False
    },
    "utokyo_pr2_opening_fridge_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "utokyo_pr2_tabletop_manipulation_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "utokyo_xarm_pick_and_place_converted_externally_to_rlds": {
        "field": ["observation", "end_effector_pose"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "robo_net": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "berkeley_mvp_converted_externally_to_rlds": {
        "field": ["observation", "pose"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "stanford_mask_vit_converted_externally_to_rlds": {
        "field": ["observation", "end_effector_pose"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "tokyo_u_lsmo_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "dlr_sara_pour_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "dlr_sara_grid_clamp_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "dlr_edan_shared_control_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "eth_agent_affordances": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "stanford_robocook_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "berkeley_fanuc_manipulation": {
        "field": ["observation", "end_effector_state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "asu_table_top_converted_externally_to_rlds": {
        "field": ["ground_truth_states", "EE"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "kaist_nonprehensile_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(14, 17),
        "reshape": False
    },
    "ucsd_pick_and_place_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "iamlab_cmu_pickup_insert_converted_externally_to_rlds": {
        "field": ["action"],
        "indices": slice(0, 3),
        "reshape": False
    },
    # --- Reshape requis (matrice homogène 4x4) ---
    "viola": {
        "field": ["observation", "ee_states"],
        "indices": None,
        "reshape": True,
    },
    "austin_buds_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(8, 24),
        "reshape": True,
    },
    "austin_sirius_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state_ee"],
        "indices": None,
        "reshape": True,
    },
    "utaustin_mutex": {
        "field": ["observation", "state"],
        "indices": slice(8, 24),
        "reshape": True,
    },
    "uiuc_d3field": {
        "field": ["observation", "state"],
        "indices": None,
        "reshape": True,
    },
}

In [ ]:
DATASET_ROBOT_INFO = {
    "fractal20220817_data":         {"robot": "Google RT-1",        "gripper": "2-finger"},
    "kuka":                         {"robot": "KUKA iiwa",          "gripper": "2-finger"},
    "taco_play":                    {"robot": "Franka Panda",        "gripper": "2-finger"},
    "jaco_play":                    {"robot": "Kinova Jaco",         "gripper": "3-finger"},
    "viola":                        {"robot": "Franka Panda",        "gripper": "2-finger"},
    "maniskill_dataset_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "stanford_kuka_multimodal_dataset_converted_externally_to_rlds": {"robot": "KUKA iiwa", "gripper": "2-finger"},
    "nyu_rot_dataset_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "stanford_hydra_dataset_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "nyu_franka_play_dataset_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "austin_sailor_dataset_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "austin_sirius_dataset_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "bc_z":                         {"robot": "Google Robot",        "gripper": "2-finger"},
    "utokyo_pr2_opening_fridge_converted_externally_to_rlds": {"robot": "PR2", "gripper": "2-finger"},
    "utokyo_pr2_tabletop_manipulation_converted_externally_to_rlds": {"robot": "PR2", "gripper": "2-finger"},
    "utokyo_xarm_pick_and_place_converted_externally_to_rlds": {"robot": "xArm", "gripper": "2-finger"},
    "robo_net":                     {"robot": "Various",             "gripper": "various"},
    "berkeley_mvp_converted_externally_to_rlds": {"robot": "xArm",  "gripper": "2-finger"},
    "stanford_mask_vit_converted_externally_to_rlds": {"robot": "Sawyer", "gripper": "2-finger"},
    "tokyo_u_lsmo_converted_externally_to_rlds": {"robot": "UR5",   "gripper": "2-finger"},
    "dlr_sara_pour_converted_externally_to_rlds": {"robot": "DLR SARA", "gripper": "2-finger"},
    "dlr_sara_grid_clamp_converted_externally_to_rlds": {"robot": "DLR SARA", "gripper": "2-finger"},
    "dlr_edan_shared_control_converted_externally_to_rlds": {"robot": "DLR EDAN", "gripper": "2-finger"},
    "eth_agent_affordances":        {"robot": "Franka Panda",        "gripper": "2-finger"},
    "stanford_robocook_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "berkeley_fanuc_manipulation":  {"robot": "FANUC",               "gripper": "2-finger"},
    "asu_table_top_converted_externally_to_rlds": {"robot": "UR5",   "gripper": "2-finger"},
    "kaist_nonprehensile_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "ucsd_pick_and_place_dataset_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "iamlab_cmu_pickup_insert_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "viola":                        {"robot": "Franka Panda",        "gripper": "2-finger"},
    "austin_buds_dataset_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "austin_sirius_dataset_converted_externally_to_rlds": {"robot": "Franka Panda", "gripper": "2-finger"},
    "utaustin_mutex":               {"robot": "Franka Panda",        "gripper": "2-finger"},
    "uiuc_d3field":                 {"robot": "Franka Panda",        "gripper": "2-finger"},
}

## Visualize the end effector final position

For all the datasets where it is available and for all the available episodes.

In [ ]:
import gc
import os
import tensorflow as tf

CACHE_DIR = "/content/endpoints_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def load_dataset(dataset_name):
    cache_path = f"{CACHE_DIR}/{dataset_name}.npy"

    if os.path.exists(cache_path):
        print(f"  ✅ {dataset_name}: already on disk")
        return

    config = DATASET_EEF_CONFIG[dataset_name]
    try:
        b = tfds.builder_from_directory(builder_dir=dataset2path(dataset_name))
        split = get_safe_split(b, max_episodes=500)

        # Charge SEULEMENT les champs nécessaires, pas les images
        read_config = tfds.ReadConfig(
            interleave_cycle_length=1,
            interleave_block_length=1,
        )
        ds = b.as_dataset(split=split, read_config=read_config)

        endpoints = []
        for episode in ds:
            last_step = None
            for step in episode["steps"]:
                last_step = step
            if last_step is not None:
                try:
                    xyz = extract_endpoint(last_step, config)
                    endpoints.append(xyz)
                except Exception:
                    pass
            del last_step
            gc.collect()

        if endpoints:
            np.save(cache_path, np.array(endpoints))
            print(f"  ✅ {dataset_name}: {len(endpoints)} episodes saved")
        else:
            print(f"  ❌ {dataset_name}: no valid endpoints")

    except Exception as e:
        print(f"  ❌ {dataset_name}: {e}")

    finally:
        gc.collect()
        tf.keras.backend.clear_session()

print("Loading all datasets...")
for name in DATASET_EEF_CONFIG:
    load_dataset(name)
print("Done!")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import os

OPENVLA_DATASETS = {
    "fractal20220817_data", "kuka", "taco_play", "jaco_play", "viola",
    "stanford_hydra_dataset_converted_externally_to_rlds",
    "austin_buds_dataset_converted_externally_to_rlds",
    "nyu_franka_play_dataset_converted_externally_to_rlds",
    "austin_sailor_dataset_converted_externally_to_rlds",
    "austin_sirius_dataset_converted_externally_to_rlds",
    "dlr_edan_shared_control_converted_externally_to_rlds",
    "iamlab_cmu_pickup_insert_converted_externally_to_rlds",
    "utaustin_mutex", "berkeley_fanuc_manipulation", "bc_z",
}

def get_endpoints(dataset_name):
    cache_path = f"/content/endpoints_cache/{dataset_name}.npy"
    if os.path.exists(cache_path):
        return np.load(cache_path)
    return None

def normalize(endpoints):
    mins = endpoints.min(axis=0)
    maxs = endpoints.max(axis=0)
    ranges = maxs - mins
    ranges[ranges == 0] = 1
    return (endpoints - mins) / ranges

# --- Widgets ---
checkboxes = {name: widgets.Checkbox(value=False, description=name, layout=widgets.Layout(width='350px'))
              for name in DATASET_EEF_CONFIG}

normalize_cb = widgets.Checkbox(value=False, description='Normalize per dataset')

view_radio = widgets.RadioButtons(
    options=['3D', 'XY', 'XZ', 'YZ'],
    value='3D',
    description='View:',
    layout=widgets.Layout(width='150px')
)

mode_radio = widgets.RadioButtons(
    options=['Combined', 'Individual'],
    value='Combined',
    description='Mode:',
    layout=widgets.Layout(width='150px')
)

output = widgets.Output()

def make_trace_2d(endpoints, name, color, axis1, axis2):
    return go.Scatter(
        x=endpoints[:, axis1],
        y=endpoints[:, axis2],
        mode='markers',
        marker=dict(size=4, opacity=0.6, color=color),
        name=name
    )

def make_trace_3d(endpoints, name, color):
    return go.Scatter3d(
        x=endpoints[:, 0],
        y=endpoints[:, 1],
        z=endpoints[:, 2],
        mode='markers',
        marker=dict(size=3, opacity=0.6, color=color),
        name=name
    )

AXIS_MAP = {'XY': (0, 1), 'XZ': (0, 2), 'YZ': (1, 2)}
AXIS_LABELS = {0: 'X', 1: 'Y', 2: 'Z'}
COLORS = [
    '#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
    '#42d4f4','#f032e6','#bfef45','#fabed4','#469990',
    '#dcbeff','#9A6324','#fffac8','#800000','#aaffc3',
    '#808000','#ffd8b1','#000075','#a9a9a9','#ffffff',
    '#000000','#e6beff','#ffe119','#4169e1','#8B0000',
    '#00CED1','#FF69B4','#32CD32','#FF8C00','#9400D3',
    '#00FA9A','#FF4500','#1E90FF',
]

def on_change(change):
    with output:
        clear_output(wait=True)
        selected = [name for name, cb in checkboxes.items() if cb.value]
        if not selected:
            print("No datasets selected.")
            return

        should_normalize = normalize_cb.value
        view = view_radio.value
        mode = mode_radio.value
        is_3d = (view == '3D')

        color_map = {name: COLORS[i % len(COLORS)] for i, name in enumerate(DATASET_EEF_CONFIG)}

        if mode == 'Combined':
            if is_3d:
                fig = go.Figure()
                fig.add_trace(go.Scatter3d(
                    x=[0], y=[0], z=[0],
                    mode='markers+text',
                    marker=dict(size=8, color='black', symbol='cross'),
                    text=['Base'], textposition='top center',
                    name='Robot Base (0,0,0)'
                ))
            else:
                a1, a2 = AXIS_MAP[view]
                fig = go.Figure()
                fig.add_trace(go.Scatter(
                    x=[0], y=[0], mode='markers+text',
                    marker=dict(size=10, color='black', symbol='cross'),
                    text=['Base'], textposition='top center',
                    name='Robot Base (0,0,0)'
                ))

            for dataset_name in selected:
                endpoints = get_endpoints(dataset_name)
                if endpoints is None:
                    print(f"  ⚠️ {dataset_name}: not on disk")
                    continue
                if should_normalize:
                    endpoints = normalize(endpoints)

                info = DATASET_ROBOT_INFO.get(dataset_name, {})
                robot = info.get("robot", "Unknown")
                tag = "✅ OpenVLA" if dataset_name in OPENVLA_DATASETS else "🔵 OOD"
                label = f"{dataset_name}<br>{robot} | {tag}"
                color = color_map[dataset_name]

                if is_3d:
                    fig.add_trace(make_trace_3d(endpoints, label, color))
                else:
                    fig.add_trace(make_trace_2d(endpoints, label, color, a1, a2))

            if is_3d:
                fig.update_layout(
                    title="EEF Endpoint Distribution — Combined 3D",
                    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', aspectmode='cube'),
                    width=1000, height=800, legend=dict(x=1.05, y=0.5)
                )
            else:
                fig.update_layout(
                    title=f"EEF Endpoint Distribution — Combined {view}",
                    xaxis_title=AXIS_LABELS[a1],
                    yaxis_title=AXIS_LABELS[a2],
                    width=1000, height=700,
                    legend=dict(x=1.05, y=0.5)
                )
            fig.show()

        else:  # Individual
            n = len(selected)
            cols = min(3, n)
            rows = (n + cols - 1) // cols

            if is_3d:
                specs = [[{"type": "scatter3d"} for _ in range(cols)] for _ in range(rows)]
            else:
                specs = [[{"type": "scatter"} for _ in range(cols)] for _ in range(rows)]

            subplot_titles = []
            for dataset_name in selected:
                info = DATASET_ROBOT_INFO.get(dataset_name, {})
                robot = info.get("robot", "Unknown")
                tag = "✅ OpenVLA" if dataset_name in OPENVLA_DATASETS else "🔵 OOD"
                subplot_titles.append(f"{dataset_name}<br>{robot} | {tag}")

            fig = make_subplots(
                rows=rows, cols=cols,
                specs=specs,
                subplot_titles=subplot_titles,
                horizontal_spacing=0.05,
                vertical_spacing=0.1
            )

            for i, dataset_name in enumerate(selected):
                row = i // cols + 1
                col = i % cols + 1

                endpoints = get_endpoints(dataset_name)
                if endpoints is None:
                    print(f"  ⚠️ {dataset_name}: not on disk")
                    continue
                if should_normalize:
                    endpoints = normalize(endpoints)

                color = color_map[dataset_name]

                if is_3d:
                    # Base marker
                    fig.add_trace(go.Scatter3d(
                        x=[0], y=[0], z=[0],
                        mode='markers',
                        marker=dict(size=6, color='black', symbol='cross'),
                        name='Base', showlegend=(i == 0)
                    ), row=row, col=col)
                    fig.add_trace(make_trace_3d(endpoints, dataset_name, color), row=row, col=col)
                else:
                    a1, a2 = AXIS_MAP[view]
                    fig.add_trace(go.Scatter(
                        x=[0], y=[0], mode='markers',
                        marker=dict(size=8, color='black', symbol='cross'),
                        name='Base', showlegend=(i == 0)
                    ), row=row, col=col)
                    fig.add_trace(make_trace_2d(endpoints, dataset_name, color, a1, a2), row=row, col=col)

            fig.update_layout(
                title=f"EEF Endpoint Distribution — Individual {'3D' if is_3d else view}",
                height=400 * rows,
                width=500 * cols,
                showlegend=False
            )
            fig.show()

# Observers
for cb in checkboxes.values():
    cb.observe(on_change, names='value')
normalize_cb.observe(on_change, names='value')
view_radio.observe(on_change, names='value')
mode_radio.observe(on_change, names='value')

# Layout
controls = widgets.VBox([
    widgets.HTML("<b>Options</b>"),
    normalize_cb,
    widgets.HTML("<hr>"),
    view_radio,
    widgets.HTML("<hr>"),
    mode_radio,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Datasets</b>"),
    widgets.VBox(list(checkboxes.values()))
])

display(widgets.HBox([controls, output]))